# Player Stats Extraction

In [ ]:
# Library Imports
import pandas as pd
import time
import numpy as np
from datetime import datetime

# Web scraping libraries
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Error handling
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException

# Data handling
import json
import os

In [7]:
# Load player data and setup browser for stats extraction
print("Loading NBA player data...")

# Load the complete player database
df_players = pd.read_csv('../data/pages/all_nba_players.csv')

print(f"Total players loaded: {len(df_players):,}")
print(f"Sample of loaded data:")
print(df_players.head())

Loading NBA player data...
Total players loaded: 5,313
Sample of loaded data:
           player_name                                         player_url  \
0       Alaa Abdelnaby  https://www.basketball-reference.com/players/a...   
1      Zaid Abdul-Aziz  https://www.basketball-reference.com/players/a...   
2  Kareem Abdul-Jabbar  https://www.basketball-reference.com/players/a...   
3   Mahmoud Abdul-Rauf  https://www.basketball-reference.com/players/a...   
4    Tariq Abdul-Wahad  https://www.basketball-reference.com/players/a...   

  letter  
0      A  
1      A  
2      A  
3      A  
4      A  


In [6]:

# Setup Chrome browser for stats extraction
print("\nSetting up Chrome browser...")
options = Options()
options.add_argument('--headless')  # Run in background
options.add_argument('--no-sandbox')  # Bypass OS security
options.add_argument('--disable-dev-shm-usage')  # Use regular storage instead of in-memory storage

service = ChromeService(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

print("Browser setup complete")
print(f"Ready to extract stats from {len(df_players)} player pages")


Setting up Chrome browser...
Browser setup complete
Ready to extract stats from 5313 player pages


In [13]:
# Find the row for Kawhi Leonard
kawhi_row = df_players[df_players['player_name'] == 'Kawhi Leonard']

# Access the 'player_url' from that row
# .values[0] is used to get the first value from the column
kawhi_url = kawhi_row['player_url'].values[0]

print(kawhi_url)

https://www.basketball-reference.com/players/l/leonaka01.html


In [16]:
# Navigate to Kawhi's page
driver.get(kawhi_url)

# Wait for the advanced stats table to load
try:
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "advanced"))
    )
    print("Advanced stats table loaded successfully")

    # Get page source and parse with BeautifulSoup
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # Find the advanced stats table
    advanced_table = soup.find('table', id='advanced')

    if advanced_table:
        print("Advanced stats table found")

        # Get table headers to understand structure
        headers = advanced_table.find('thead').find_all('th')
        header_names = [th.get('data-stat', th.text.strip()) for th in headers]

        print(f"Table headers found: {len(header_names)}")
        print(f"Header names: {header_names[:10]}...")  # Show first 10

    else:
        print("Advanced stats table not found")

except TimeoutException:
    print("Timeout waiting for advanced stats table")
    print(f"Current URL: {driver.current_url}")

Advanced stats table loaded successfully
Advanced stats table found
Table headers found: 29
Header names: ['year_id', 'age', 'team_name_abbr', 'comp_name_abbr', 'pos', 'games', 'games_started', 'mp', 'per', 'ts_pct']...


In [19]:
# Extract Kawhi's season-by-season stats
print("Extracting Kawhi's season stats...")

# Get all data rows from the advanced table
rows = advanced_table.find('tbody').find_all('tr')

# Filter out header rows and totals rows
data_rows = []
for row in rows:
    # Skip rows that are headers or don't have season data
    season_cell = row.find('th', {'data-stat': 'year_id'})
    if season_cell and season_cell.text.strip() and not season_cell.text.strip().startswith('Career'):
        data_rows.append(row)

print(f"Found {len(data_rows)} seasons of data")

# Extract the specific columns we need
kawhi_seasons = []

target_stats = ['year_id', 'age', 'team_name_abbr', 'games', 'mp', 'per', 'bpm', 'vorp', 'usg_pct', 'ws', 'ws_per_48']

for row in data_rows:
    season_data = {}
    
    # Extract each target stat
    for stat in target_stats:
        cell = row.find(['th', 'td'], {'data-stat': stat})
        if cell:
            value = cell.text.strip()
            season_data[stat] = value if value else None
        else:
            season_data[stat] = None
    
    kawhi_seasons.append(season_data)

# Convert to DataFrame for easy viewing
kawhi_df = pd.DataFrame(kawhi_seasons)

print("\nKawhi Leonard's extracted stats:")
print(kawhi_df)

# Check for any missing key stats
print(f"\nData completeness check:")
for stat in ['per', 'bpm', 'ws', 'ws_per_48']:
    missing = kawhi_df[stat].isna().sum()
    total = len(kawhi_df)
    print(f"{stat.upper()}: {total - missing}/{total} seasons available")

Extracting Kawhi's season stats...
Found 14 seasons of data

Kawhi Leonard's extracted stats:
    year_id   age         team_name_abbr games    mp   per   bpm  vorp  \
0   2011-12    20                    SAS    64  1534  16.6   3.4   2.1   
1   2012-13    21                    SAS    58  1810  16.4   3.1   2.3   
2   2013-14    22                    SAS    66  1923  19.4   5.1   3.5   
3   2014-15    23                    SAS    64  2033  22.0   5.9   4.1   
4   2015-16    24                    SAS    72  2380  26.0   9.1   6.7   
5   2016-17    25                    SAS    74  2474  27.6   9.4   7.1   
6   2017-18    26                    SAS     9   210  26.0  10.1   0.6   
7   2018-19    27                    TOR    60  2040  25.8   7.2   4.7   
8   2019-20    28                    LAC    57  1848  26.9   8.9   5.1   
9   2020-21    29                    LAC    52  1773  26.0   7.3   4.2   
10  2021-22  None  Did not play - injury  None  None  None  None  None   
11  2022-23    31 

In [20]:
# Extract stats from 20 random players to test our pipeline
print("Testing extraction pipeline with 20 random players...")

# Select 20 random players from our database
random_players = df_players.sample(n=20, random_state=42)
print(f"Selected {len(random_players)} random players for testing")

# Define our target stats
target_stats = ['year_id', 'age', 'team_name_abbr', 'games', 'mp', 'per', 'bpm', 'vorp', 'ws', 'ws_per_48']

# Store all extracted data
all_player_stats = []
successful_extractions = 0
failed_extractions = 0

for idx, (_, player) in enumerate(random_players.iterrows(), 1):
    player_name = player['player_name']
    player_url = player['player_url']
    
    print(f"\n[{idx}/20] Extracting: {player_name}")
    
    try:
        # Navigate to player page
        driver.get(player_url)
        
        # Wait for advanced stats table
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "advanced"))
        )
        
        # Parse the page
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        advanced_table = soup.find('table', id='advanced')
        
        if advanced_table:
            # Get data rows (exclude headers and totals)
            rows = advanced_table.find('tbody').find_all('tr')
            data_rows = []
            
            for row in rows:
                season_cell = row.find('th', {'data-stat': 'year_id'})
                if season_cell and season_cell.text.strip() and not season_cell.text.strip().startswith('Career'):
                    data_rows.append(row)
            
            # Extract stats for each season
            for row in data_rows:
                season_data = {'player_name': player_name}
                
                for stat in target_stats:
                    cell = row.find(['th', 'td'], {'data-stat': stat})
                    if cell:
                        value = cell.text.strip()
                        season_data[stat] = value if value else None
                    else:
                        season_data[stat] = None
                
                all_player_stats.append(season_data)
            
            print(f"  ✓ Extracted {len(data_rows)} seasons")
            successful_extractions += 1
            
        else:
            print(f"  ✗ No advanced stats table found")
            failed_extractions += 1
    
    except Exception as e:
        print(f"  ✗ Error: {str(e)}")
        failed_extractions += 1
    
    # Rate limiting - 3 second delay
    time.sleep(3)

# Convert to DataFrame
df_extracted = pd.DataFrame(all_player_stats)

print(f"\n=== EXTRACTION SUMMARY ===")
print(f"Successful players: {successful_extractions}/20")
print(f"Failed players: {failed_extractions}/20")
print(f"Total seasons extracted: {len(df_extracted)}")

if len(df_extracted) > 0:
    print(f"\nSample of extracted data:")
    print(df_extracted.head())
    
    # Save test data
    df_extracted.to_csv('../data/pages/test_player_stats_20.csv', index=False)
    print(f"\nSaved test data to '../data/pages/test_player_stats_20.csv'")
else:
    print("\nNo data extracted - check for errors above")

Testing extraction pipeline with 20 random players...
Selected 20 random players for testing

[1/20] Extracting: Ed Dahler
  ✓ Extracted 2 seasons

[2/20] Extracting: Zach Randolph
  ✓ Extracted 19 seasons

[3/20] Extracting: Jim Kissane
  ✓ Extracted 1 seasons

[4/20] Extracting: Andrés Guibert
  ✓ Extracted 2 seasons

[5/20] Extracting: Chris Gatling
  ✓ Extracted 20 seasons

[6/20] Extracting: Dan Gadzuric
  ✓ Extracted 12 seasons

[7/20] Extracting: Ed Beach
  ✓ Extracted 3 seasons

[8/20] Extracting: Phil Wagner
  ✓ Extracted 1 seasons

[9/20] Extracting: Danny Manning
  ✓ Extracted 17 seasons

[10/20] Extracting: Dave Hoppen
  ✓ Extracted 11 seasons

[11/20] Extracting: Mouhamed Gueye
  ✓ Extracted 2 seasons

[12/20] Extracting: Darius Johnson-Odom
  ✓ Extracted 2 seasons

[13/20] Extracting: Leroy Combs
  ✓ Extracted 1 seasons

[14/20] Extracting: Amile Jefferson
  ✓ Extracted 2 seasons

[15/20] Extracting: Jerry Pender
  ✓ Extracted 2 seasons

[16/20] Extracting: Reggie Royals


In [ ]:
# Full dataset extraction - All NBA players
print("Starting full dataset extraction...")
print(f"Total players to process: {len(df_players)}")

# Estimate time based on actual performance (20 players = 3m 54.6s)
actual_rate = 234.6 / 20  # seconds per player from test run
estimated_seconds = len(df_players) * actual_rate
estimated_minutes = estimated_seconds / 60
estimated_hours = estimated_minutes / 60
print(f"Estimated time: {estimated_minutes:.0f} minutes ({estimated_hours:.1f} hours)")
print(f"Based on test run: 20 players in 3m 54.6s = {actual_rate:.1f}s per player")

# Define target stats
target_stats = ['year_id', 'age', 'team_name_abbr', 'games', 'mp', 'per', 'bpm', 'vorp', 'ws', 'ws_per_48']

# Store all extracted data
all_player_stats = []
successful_extractions = 0
failed_extractions = 0
failed_players = []

# Track progress
start_time = datetime.now()
save_interval = 100  # Save progress every 100 players

print(f"\nStarting extraction at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("Progress will be saved every 100 players...")

for idx, (_, player) in enumerate(df_players.iterrows(), 1):
    player_name = player['player_name']
    player_url = player['player_url']

    # Progress indicator
    if idx % 50 == 0 or idx <= 10:
        elapsed = datetime.now() - start_time
        rate = idx / elapsed.total_seconds() * 60  # players per minute
        remaining = (len(df_players) - idx) / rate if rate > 0 else 0
        print(f"[{idx}/{len(df_players)}] {player_name} | Rate: {rate:.1f}/min | ETA: {remaining:.0f}min")

    try:
        # Navigate to player page
        driver.get(player_url)

        # Wait for advanced stats table
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "advanced"))
        )

        # Parse the page
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        advanced_table = soup.find('table', id='advanced')

        if advanced_table:
            # Get data rows (exclude headers and totals)
            rows = advanced_table.find('tbody').find_all('tr')
            data_rows = []

            for row in rows:
                season_cell = row.find('th', {'data-stat': 'year_id'})
                if season_cell and season_cell.text.strip() and not season_cell.text.strip().startswith('Career'):
                    data_rows.append(row)

            # Extract stats for each season
            for row in data_rows:
                season_data = {'player_name': player_name}

                for stat in target_stats:
                    cell = row.find(['th', 'td'], {'data-stat': stat})
                    if cell:
                        value = cell.text.strip()
                        season_data[stat] = value if value else None
                    else:
                        season_data[stat] = None

                all_player_stats.append(season_data)

            successful_extractions += 1

        else:
            failed_extractions += 1
            failed_players.append({'player': player_name, 'reason': 'No advanced table'})

    except Exception as e:
        failed_extractions += 1
        failed_players.append({'player': player_name, 'reason': str(e)})

    # Save progress every 100 players
    if idx % save_interval == 0:
        df_temp = pd.DataFrame(all_player_stats)
        df_temp.to_csv(f'../data/pages/nba_stats_progress_{idx}.csv', index=False)
        print(f"  -> Progress saved: {len(all_player_stats)} seasons extracted")

    # Rate limiting - 3 second delay
    time.sleep(3)

# Final save
df_final = pd.DataFrame(all_player_stats)
df_final.to_csv('../data/pages/nba_all_player_stats.csv', index=False)

# Save failed players log
if failed_players:
    df_failed = pd.DataFrame(failed_players)
    df_failed.to_csv('../data/pages/failed_extractions.csv', index=False)

# Final summary
end_time = datetime.now()
total_time = end_time - start_time

print(f"\n=== EXTRACTION COMPLETE ===")
print(f"Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"End time: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total time: {total_time}")
print(f"Successful players: {successful_extractions}/{len(df_players)} ({successful_extractions/len(df_players)*100:.1f}%)")
print(f"Failed players: {failed_extractions}")
print(f"Total seasons extracted: {len(df_final)}")
print(f"Average seasons per player: {len(df_final)/successful_extractions:.1f}")
print(f"\nFinal dataset saved to: '../data/pages/nba_all_player_stats.csv'")

# Close browser
driver.quit()
print("Browser closed.")